In [1]:
#################
#Input parameters
#################

Unrest = True # True if you want to compute probabilities from monitoring data, False if you want to compute probabilities from long-term data
n_sample = 100000 # length of the sample of the probability distributions
dt = 1 # time window, lower or equal than 1
prc = [10, 50, 90] # 3 percentiles of the distributions you want as output

if Unrest == True: #fill if you want to compute the short-term probability
    
    #Anomaly Function Parameters
    a = 0.9
    lambda_short_term = 1

    #Monitoring file (path or file name if it's in the same directory of the code)
    Monitoring = 'Monitoring.xlsx'

if Unrest == False: #fill if you want to compute the long-term probability
    
    #Long-term Unrest
    n1 = 306
    x1 = 7
    mean_prior_1 = 0.5
    lambda_prior_1 = 1

    #Long-term Magmatic Unrest
    n2 = 7
    x2 = 1
    mean_prior_2 = 0.5
    lambda_prior_2 = 1

    #Long-term Eruption
    n3 = 3.7
    x3 = 0
    mean_prior_3 = 0.5
    lambda_prior_3 = 1

In [2]:
import functions_BET as fb
import numpy as np
import pandas as pd

In [3]:
#Long_term

if Unrest == False:

    Phi_prior1 = mean_prior_1
    lam_prior1 = lambda_prior_1
    Phi_prior2 = mean_prior_2
    lam_prior2 = lambda_prior_2
    Phi_prior3 = mean_prior_3
    lam_prior3 = lambda_prior_3
    
    phi1 = fb.long_term_probability(Phi_prior1, lam_prior1, n1, x1, n_sample, dt)
    phi2 = fb.long_term_probability(Phi_prior2, lam_prior2, n2, x2, n_sample, dt)
    phi3 = fb.long_term_probability(Phi_prior3, lam_prior3, n3, x3, n_sample, dt)

    mean_unrest, prc_unrest, mean_magma, prc_magma, mean_eruption, prc_eruption = fb.absolute_probability(phi2, phi3, prc, phi1)

    #Output file
    
    Table = np.zeros((3, 4))
    Table[0][0] = mean_unrest
    Table[0][1::] = prc_unrest
    Table[1][0] = mean_magma
    Table[1][1::] = prc_magma
    Table[2][0] = mean_eruption
    Table[2][1::] = prc_eruption
    cols = ['Mean', f'{prc[0]}-th Percentile', f'{prc[1]}-th Percentile', f'{prc[2]}-th Percentile']
    Table = pd.DataFrame(Table, columns = cols)
    Table.index = ['Probability of Unrest', 'Probability of Magmatic Unrest', 'Probability of Eruption']
    Table.to_excel('long-term Eruption Forecasting.xlsx', index = True)    

#Short_term 

if Unrest == True:

    M2 = pd.read_excel(Monitoring, sheet_name = 'Magmatic Unrest').to_numpy()
    P2 = pd.read_excel(Monitoring, sheet_name = 'Magmatic Unrest - Parameters').to_numpy()
    M3 = pd.read_excel(Monitoring, sheet_name = 'Eruption').to_numpy()
    P3 = pd.read_excel(Monitoring, sheet_name = 'Eruption - Parameters').to_numpy()
    
    phi2 = fb.short_term_probability(M2, P2, lambda_short_term, a, n_sample)
    phi3 = fb.short_term_probability(M3, P3, lambda_short_term, a, n_sample)

    if dt < 1:
        phi3 = fb.eruption_probability_rescale_st(phi3, dt)
        
    mean_magma, prc_magma, mean_eruption, prc_eruption = fb.absolute_probability(phi2, phi3, prc)

    #Output_file
    
    Table = np.zeros((8, len(M2)))
    Table[0] = mean_magma
    Table[1:4] = prc_magma.T
    Table[4] = mean_eruption
    Table[5:8] = prc_eruption.T
    Table = Table.T
    cols = ['Mean Magmatic Unrest', f'{prc[0]}-th Percentile Magmatic Unrest', f'{prc[1]}-th Percentile Magmatic Unrest', f'{prc[2]}-th Percentile Magmatic Unrest', 'Mean Eruption', f'{prc[0]}-th percentile Eruption', f'{prc[1]}-th percentile Eruption', f'{prc[2]}-th percentile Eruption']
    Table = pd.DataFrame(Table, columns = cols)
    Table.index = Table.index + 1
    Table.to_excel('short-term Eruption Forecasting.xlsx', index = False)